# 03 — 2048 Mass–Spring Cubes in Parallel

## 목표

단일 cube simulator를 **2048개 환경**으로 확장한다.

핵심 설계:

- state shape: `[num_envs, 8, 3]`
- action shape: `[num_envs, 12]`
- Warp 내부 state는 flatten된 `[num_envs*8]` `wp.vec3`
- thread 하나가 particle 하나를 담당
- `wp.to_torch()`로 state를 복사 없이 PyTorch tensor로 읽음

2048개 cube × 8 particles = **16,384 particle threads**가 한 번에 실행된다.

In [ ]:
# Colab 권장: Runtime > Change runtime type > T4 GPU
!nvidia-smi

# Warp 1.17.0의 PyPI Linux wheel은 CUDA 12.9 runtime 기반이라
# Colab의 일반적인 NVIDIA driver에서 호환성이 좋다.
%pip -q install "warp-lang==1.17.0"

import warp as wp
wp.init()
wp.print_diagnostics()

DEVICE = "cuda:0" if wp.is_cuda_available() else "cpu"
print("Selected Warp device:", DEVICE)

## 1. Topology

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warp as wp

SIDE = 0.4

def make_cube_topology(side=SIDE, z0=0.45):
    # bit pattern 순서의 8 vertices
    verts = np.array([
        [0,0,0], [1,0,0], [0,1,0], [1,1,0],
        [0,0,1], [1,0,1], [0,1,1], [1,1,1],
    ], dtype=np.float32)

    verts *= side
    verts[:, 0] -= side * 0.5
    verts[:, 1] -= side * 0.5
    verts[:, 2] += z0

    springs = []
    actuator_index = []
    actuator_count = 0

    # 모든 8C2 = 28 pair를 연결:
    # 12 edge + 12 face diagonal + 4 body diagonal
    for i in range(8):
        for j in range(i + 1, 8):
            d = np.linalg.norm(verts[j] - verts[i])
            springs.append((i, j))

            if np.isclose(d, side, atol=1e-5):
                actuator_index.append(actuator_count)
                actuator_count += 1
            else:
                actuator_index.append(-1)

    spring_i = np.array([s[0] for s in springs], dtype=np.int32)
    spring_j = np.array([s[1] for s in springs], dtype=np.int32)
    rest = np.array(
        [np.linalg.norm(verts[j] - verts[i]) for i, j in springs],
        dtype=np.float32
    )

    return verts, spring_i, spring_j, rest, np.array(actuator_index, np.int32)

BASE_X, SPRING_I, SPRING_J, BASE_REST, ACT_IDX = make_cube_topology()

print("particles:", len(BASE_X))
print("springs:", len(SPRING_I))
print("actuated edge springs:", np.sum(ACT_IDX >= 0))

## 2. Vectorized Warp kernels

In [ ]:
@wp.kernel
def reset_vec(
    base_x: wp.array(dtype=wp.vec3),
    x: wp.array(dtype=wp.vec3),
    v: wp.array(dtype=wp.vec3),
    env_spacing: float,
):
    tid = wp.tid()
    env = tid // 8
    p = tid - env * 8

    # visualization/debug 시 서로 다른 위치에 놓고 싶으면 y offset을 줄 수 있다.
    # RL 상태에서는 translation invariance를 쓰므로 기본은 같은 위치에 겹쳐 둔다.
    x[tid] = base_x[p]
    v[tid] = wp.vec3(0.0, 0.0, 0.0)


@wp.kernel
def compute_force_vec(
    x: wp.array(dtype=wp.vec3),
    v: wp.array(dtype=wp.vec3),
    spring_i: wp.array(dtype=wp.int32),
    spring_j: wp.array(dtype=wp.int32),
    rest: wp.array(dtype=float),
    actuator_index: wp.array(dtype=wp.int32),
    actions: wp.array2d(dtype=float),
    force: wp.array(dtype=wp.vec3),
    n_springs: int,
    mass: float,
    k: float,
    c: float,
    act_amp: float,
    ground_k: float,
    ground_c: float,
    friction: float,
):
    tid = wp.tid()
    env = tid // 8
    p = tid - env * 8
    base = env * 8

    xp = x[tid]
    vp = v[tid]
    f = wp.vec3(0.0, 0.0, -9.81 * mass)

    for s in range(n_springs):
        i = spring_i[s]
        j = spring_j[s]

        if p == i or p == j:
            xi = x[base + i]
            xj = x[base + j]
            vi = v[base + i]
            vj = v[base + j]

            d = xj - xi
            L = wp.length(d)
            n = d / (L + 1.0e-8)

            L0 = rest[s]
            aidx = actuator_index[s]
            if aidx >= 0:
                L0 = L0 * (1.0 + act_amp * actions[env, aidx])

            rel = wp.dot(vj - vi, n)
            mag = k * (L - L0) + c * rel
            fs = mag * n

            if p == i:
                f = f + fs
            else:
                f = f - fs

    if xp[2] < 0.0:
        f = f + wp.vec3(0.0, 0.0, -ground_k * xp[2])

        if vp[2] < 0.0:
            f = f + wp.vec3(0.0, 0.0, -ground_c * vp[2])

        f = f + wp.vec3(-friction * vp[0], -friction * vp[1], 0.0)

    force[tid] = f


@wp.kernel
def integrate_vec(
    x: wp.array(dtype=wp.vec3),
    v: wp.array(dtype=wp.vec3),
    force: wp.array(dtype=wp.vec3),
    mass: float,
    dt: float,
):
    tid = wp.tid()

    a = force[tid] / mass
    v_new = v[tid] + a * dt
    x_new = x[tid] + v_new * dt

    v[tid] = v_new
    x[tid] = x_new

## 3. Vectorized simulator

In [ ]:
import torch

TORCH_DEVICE = torch.device("cuda" if DEVICE.startswith("cuda") else "cpu")

class VectorizedWarpCube:
    def __init__(
        self,
        num_envs=2048,
        device=DEVICE,
        dt=0.0025,
        substeps=4,
        mass=0.15,
        k=180.0,
        c=2.0,
        act_amp=0.20,
        ground_k=1500.0,
        ground_c=15.0,
        friction=2.0,
    ):
        self.num_envs = num_envs
        self.device = device
        self.dt = dt
        self.substeps = substeps
        self.mass = mass
        self.k = k
        self.c = c
        self.act_amp = act_amp
        self.ground_k = ground_k
        self.ground_c = ground_c
        self.friction = friction

        self.n_particles = 8
        self.n_springs = len(SPRING_I)
        self.n_act = int(np.sum(ACT_IDX >= 0))

        self.base_x = wp.array(BASE_X, dtype=wp.vec3, device=device)
        self.spring_i = wp.array(SPRING_I, dtype=wp.int32, device=device)
        self.spring_j = wp.array(SPRING_J, dtype=wp.int32, device=device)
        self.rest = wp.array(BASE_REST, dtype=float, device=device)
        self.actuator_index = wp.array(ACT_IDX, dtype=wp.int32, device=device)

        N = num_envs * 8
        self.x = wp.zeros(N, dtype=wp.vec3, device=device)
        self.v = wp.zeros(N, dtype=wp.vec3, device=device)
        self.force = wp.zeros(N, dtype=wp.vec3, device=device)

        self.reset()

    def reset(self):
        wp.launch(
            reset_vec,
            dim=self.num_envs * 8,
            inputs=[self.base_x, self.x, self.v, 0.0],
            device=self.device,
        )

    def state_torch(self):
        # zero-copy view: [E*8,3] -> [E,8,3]
        x_t = wp.to_torch(self.x).view(self.num_envs, 8, 3)
        v_t = wp.to_torch(self.v).view(self.num_envs, 8, 3)
        return x_t, v_t

    def observation(self):
        x_t, v_t = self.state_torch()

        com = x_t.mean(dim=1)
        com_v = v_t.mean(dim=1)
        rel = x_t - com[:, None, :]

        obs = torch.cat(
            [
                rel.reshape(self.num_envs, -1),   # 24
                v_t.reshape(self.num_envs, -1),   # 24
                com[:, 2:3],                      # 1
                com_v,                            # 3
            ],
            dim=1,
        )
        return obs

    def step(self, actions_t):
        # actions_t: [E, 12], same GPU device, contiguous float32
        actions_t = actions_t.contiguous().to(dtype=torch.float32)
        actions_w = wp.from_torch(actions_t, requires_grad=False)

        x_before, _ = self.state_torch()
        com_x_before = x_before[:, :, 0].mean(dim=1).clone()

        for _ in range(self.substeps):
            wp.launch(
                compute_force_vec,
                dim=self.num_envs * 8,
                inputs=[
                    self.x, self.v,
                    self.spring_i, self.spring_j,
                    self.rest, self.actuator_index,
                    actions_w, self.force,
                    self.n_springs,
                    self.mass, self.k, self.c, self.act_amp,
                    self.ground_k, self.ground_c, self.friction,
                ],
                device=self.device,
            )
            wp.launch(
                integrate_vec,
                dim=self.num_envs * 8,
                inputs=[self.x, self.v, self.force, self.mass, self.dt],
                device=self.device,
            )

        x_after, _ = self.state_torch()
        com = x_after.mean(dim=1)

        dx = com[:, 0] - com_x_before
        energy = actions_t.square().mean(dim=1)
        low_height = torch.relu(0.22 - com[:, 2])

        reward = 10.0 * dx - 0.01 * energy - 0.5 * low_height
        obs = self.observation()

        return obs, reward, {
            "com_x": com[:, 0],
            "com_z": com[:, 2],
            "energy": energy,
        }

## 4. 2048 environments smoke test

In [ ]:
NUM_ENVS = 2048

envs = VectorizedWarpCube(num_envs=NUM_ENVS)

obs = envs.observation()
print("obs shape:", obs.shape)       # [2048, 52]
print("action dim:", envs.n_act)     # 12
print("torch device:", obs.device)

actions = torch.zeros((NUM_ENVS, envs.n_act), device=obs.device)
obs, reward, info = envs.step(actions)

print("reward shape:", reward.shape)
print("mean reward:", reward.mean().item())

## 5. Random action stress test

모든 환경이 서로 다른 random action을 받는다.

In [ ]:
envs.reset()

for t in range(200):
    actions = 2.0 * torch.rand(
        (NUM_ENVS, envs.n_act),
        device=TORCH_DEVICE
    ) - 1.0

    obs, reward, info = envs.step(actions)

print("mean COM x:", info["com_x"].mean().item())
print("mean COM z:", info["com_z"].mean().item())
print("finite obs:", torch.isfinite(obs).all().item())

## 6. 처리량 benchmark

첫 kernel launch는 JIT compile을 포함하므로 warm-up 뒤 timing 한다.

In [ ]:
envs.reset()
actions = torch.zeros((NUM_ENVS, envs.n_act), device=TORCH_DEVICE)

# warm-up
for _ in range(10):
    envs.step(actions)

if TORCH_DEVICE.type == "cuda":
    torch.cuda.synchronize()

import time
tic = time.perf_counter()

N_STEPS = 300
for _ in range(N_STEPS):
    actions.uniform_(-1.0, 1.0)
    envs.step(actions)

if TORCH_DEVICE.type == "cuda":
    torch.cuda.synchronize()

elapsed = time.perf_counter() - tic

physics_env_steps = NUM_ENVS * N_STEPS
print(f"{physics_env_steps:,} vector env-steps in {elapsed:.3f}s")
print(f"throughput: {physics_env_steps/elapsed:,.0f} env-steps/s")

## 7. 한 환경만 꺼내서 보기

2048개를 겹쳐서 simulate하지만, 개별 환경의 좌표는 독립적이다.

In [ ]:
x_t, v_t = envs.state_torch()

env_id = 0
x0 = x_t[env_id].detach().cpu().numpy()

def plot_one_cube(x, title="one of 2048 cubes"):
    fig = plt.figure(figsize=(6,5))
    ax = fig.add_subplot(111, projection="3d")
    for i, j in zip(SPRING_I, SPRING_J):
        p, q = x[i], x[j]
        ax.plot([p[0],q[0]], [p[1],q[1]], [p[2],q[2]], alpha=0.3)
    ax.scatter(x[:,0], x[:,1], x[:,2], s=55)
    ax.set_zlim(-0.05, 0.9)
    ax.set_title(title)
    plt.show()

plot_one_cube(x0)

## 8. Scaling Lab

아래 값으로 throughput을 비교해보자.

```python
num_envs = [64, 256, 512, 1024, 2048, 4096]
```

### 질문
1. 환경 수를 2배로 늘리면 실행시간도 정확히 2배가 되는가?
2. 작은 환경 수에서는 왜 GPU utilization이 낮을 수 있는가?
3. PyTorch tensor와 Warp array 사이에 매 step CPU 복사가 있다면 어떤 문제가 생길까?

다음 notebook에서는 이 2048개 환경을 PPO rollout collector로 그대로 사용한다.